duck duck go search agent 


In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [ ]:
search_tool = DuckDuckGoSearchRun()

search_tool.invoke("what is the name of amazon owner ?")




"Jeffrey Preston Bezos (/ ˈbeɪzoʊs / BAY-zohss; [2] né Jorgensen; born January 12, 1964) is an American businessman best known as the founder, executive chairman, and former president and CEO ofAmazon, the world's largest e-commerce and cloud computing company. Dec 16, 2025 ·Jeff Bezos is the topAmazonshareholder with roughly $166 billion worth of shares, followed by institutional investors. 4 days ago ·Jeff Bezos founded e-commerce giantAmazonin 1994 out of his Seattle garage. Bezos stepped down as CEO to become executive chairman in 2021. He owns 8% of the company.Oct 1, 2025 ·Discover whoownsAmazonand its largest shareholders. Breakdown ofAmazon's biggest shareholders including Jeff Bezos, institutional investors.Jeff Bezos foundedAmazon.com in 1994. Amazon’s mission is to be Earth's most customer-centric company.May 23, 2023 ·Entrepreneur and e-commerce pioneerJeff Bezos is the founder and executive chair of Amazon, owner of The Washington Post, and founder of the space exploration

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

llm.invoke("hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d0441-d1e1-7613-be97-fa9f08ec9b6a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 10, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}})

Now we have llm and search tool



now creating agent = llm + tool

In [ ]:
from langchain.agents import create_agent


search_agent = create_agent(
    model= llm,
    tools=[search_tool],
    system_prompt="You are a duckduckgo search agent tool , and only use the provided tool for answer user query ",
)


result = search_agent.invoke({'messages':[{'role': 'user', 'content': 'who is the new mayor of new york 2026'}]})

result

{'messages': [HumanMessage(content='who is the new mayor of new york 2026', additional_kwargs={}, response_metadata={}, id='a5cd3ef2-3f08-45ee-8bf4-080be1723e88'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'duckduckgo_search', 'arguments': '{"query": "new mayor of new york 2026"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d0451-8d87-7a92-bb0c-b055e27a913c-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'new mayor of new york 2026'}, 'id': '455e55fa-6c31-41c5-ae49-9b36b2ce206f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 100, 'output_tokens': 26, 'total_tokens': 126, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="He won the Democratic primary in June 2025, defeating former governor Andrew Cuomo in an upset, and was electedmayorin the November general election. He isNewYorkCity

In [16]:
result['messages'][-1].content

"The new mayor of New York is Zohran Mamdani. He was sworn in on January 1, 2026. He is the city's first Muslim and first Asian American mayor, as well as its first mayor of South Asian descent. He is also the youngest leader of New York City in over a century."

adding weather tool to our search agent as well 


In [ ]:
import requests

city =' hyderabad'
@tool
def get_weather_data(city : str) -> str :
    """
    this funtion fetches the current temperature of the given city 
    """
    url = f"http://api.weatherapi.com/v1/current.json?key=9a590e4d07ee4f77a4a52828250409?q={city}"


    response = requests.get(url)

    return response.json()

# data['current']['temp_c'] 



#    params = {
#     "key": API_KEY,
#     "q": "Hyderabad"
# }

# requests.get(base, params=params)

In [25]:
search_agent = create_agent(
    model= llm,
    tools=[search_tool,get_weather_data],
    system_prompt="You are a duckduckgo search agent tool , and only use the provided tool for answer user query ",
)


result = search_agent.invoke({'messages':[{'role': 'user', 'content': "name one popular city in south india and give that city's current temperature"}]})

result

{'messages': [HumanMessage(content="name one popular city in south india and give that city's current temperature", additional_kwargs={}, response_metadata={}, id='be6972a4-0e8d-4fa5-bf4f-576e8eee2a40'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'duckduckgo_search', 'arguments': '{"query": "popular cities in south india"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d0472-9fdc-7b60-ad55-ca2d4103dafe-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'popular cities in south india'}, 'id': 'e6ab81c8-840e-48d3-8d79-44f9a9403d27', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 147, 'output_tokens': 21, 'total_tokens': 168, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="January 27, 2026 -South India contributes 30% of India's GDP with a higher per capita income and lower debt-to-GDP ratio

In [26]:
result['messages'][-1].content

'Chennai is a popular city in South India, and its current temperature is 28.2°C.'